# RO2 — LLM/SLM Hybrid: Multiplicative Sentiment Gating + Calibrated Selective Forecasting

**Research objective (RO2):** *"To design a hybrid framework that combines large language models (LLMs),
sentiment analysis techniques, and conventional financial indicators, focusing on optimizing deep learning
architectures for real-time stock market predictions."*

**Base papers:**
- *Multiplicative Information Gating: Decomposing news into novelty, materiality, and polarity for calibrated
  selective forecasting* — an LLM/SLM reads each headline and produces three scores (polarity, novelty,
  materiality) that are combined **multiplicatively** (not averaged) so a near-zero factor vetoes the signal;
  the resulting gated feature feeds a **calibrated, probabilistic** forecaster with a **conviction gate** that
  can abstain.
- *Enhancing Financial Forecasting via Multimodal Learning and Sliding-Window Metaheuristic (PSO) Optimization*
  — contributes the "optimize the architecture automatically" angle, reproduced here as a small hand-rolled
  particle-swarm search over the forecaster's hyperparameters.

**What this notebook does (simplified, real-data version):**
1. Pulls **real, current** news headlines per stock (Yahoo Finance) — free APIs only return a recent window
   (typically the trailing few weeks), so this notebook pools **~35 NIFTY50 names** (up from a smaller pilot
   universe) to get enough real, sentiment-covered rows for a meaningful demo. This is disclosed, not hidden.
2. Scores each headline with a small language model (**FinBERT**, financial-domain BERT) for polarity, a
   TF-IDF novelty score against the ticker's own recent headline history, and a keyword-based materiality
   score — then multiplies them into one gated signal, exactly as the paper's mechanism.
3. Predicts the **21-trading-day ("~30-day swing") forward return**, not next-day — this project's own
   production ledger found no usable edge at 1-day horizons (next-period direction ≈ 50%, ranking AUC ≈ 0.47)
   and that real, gate-able edge shows up at a ~30-day swing horizon instead. This notebook targets the same
   horizon so its findings are structurally comparable to that result, not a different, easier question.
4. Fits a **quantile forecaster** (LightGBM, quantiles 0.05 / 0.5 / 0.95) on technical indicators + the gated
   sentiment signal, then runs a **two-stage recalibration**: (a) an isotonic-regression bias correction on
   the median forecast (fixes systematic over/under-shoot — the compressed analogue of this project's
   production "stacking domain mismatch" fix), then (b) **conformalized quantile calibration** (Romano et al.
   2019) to fix interval coverage. A **conviction/abstention gate** driven by predicted interval width follows.
5. Runs a tiny hand-rolled **PSO** over the forecaster's hyperparameters (learning rate, tree complexity) —
   the "optimize the architecture" piece from the second paper — minimizing validation pinball loss.
6. Reports empirical interval coverage before/after calibration, and directional hit-rate at a high-conviction
   operating point **with a Wilson confidence interval**, against the no-gate baseline — the same structure
   (base rate vs. gated hit rate vs. abstention level, with a CI) as this project's own published production
   result (58.0% base vs 60.6% gated at ~94% abstention, 95% CI 58.8-62.4, on a 619-621 row live ledger). We
   do **not** expect to reproduce those exact numbers on a demo-scale, free-data pull — the point is a
   structurally comparable, honestly-reported result on a much smaller real sample.

Runtime: ~2-4 minutes on a free Colab T4 (mostly the FinBERT pass; LightGBM/PSO steps are seconds).

In [ ]:
!pip -q install yfinance==0.2.* lightgbm --upgrade
import warnings; warnings.filterwarnings("ignore")
print("done")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yfinance as yf
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 7
np.random.seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else -1
print("gpu available:", torch.cuda.is_available())

TICKERS = ["RELIANCE.NS","TCS.NS","HDFCBANK.NS","INFY.NS","ICICIBANK.NS","ITC.NS","LT.NS","SBIN.NS",
           "BHARTIARTL.NS","HINDUNILVR.NS","KOTAKBANK.NS","AXISBANK.NS","MARUTI.NS","SUNPHARMA.NS",
           "TATASTEEL.NS","WIPRO.NS","ONGC.NS","NTPC.NS","POWERGRID.NS","ULTRACEMCO.NS","ASIANPAINT.NS",
           "BAJFINANCE.NS","BAJAJFINSV.NS","TITAN.NS","NESTLEIND.NS","HCLTECH.NS","ADANIENT.NS",
           "ADANIPORTS.NS","COALINDIA.NS","DRREDDY.NS","GRASIM.NS","HEROMOTOCO.NS","INDUSINDBK.NS",
           "JSWSTEEL.NS","M&M.NS"]
START = "2015-01-01"
HORIZON = 21   # ~30 calendar days -- the swing horizon where this project's production ledger found real edge

### 1. Real prices + technical indicators (backbone features for the forecaster)

In [ ]:
def fetch_prices(ticker, start=START):
    df = yf.download(ticker, start=start, progress=False, auto_adjust=True)
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    return df.dropna(how="all")

def add_technical(df):
    out = df.copy()
    out["ret1"] = out["Close"].pct_change()
    out["ret5"] = out["Close"].pct_change(5)
    delta = out["Close"].diff()
    up = delta.clip(lower=0).rolling(14).mean()
    down = (-delta.clip(upper=0)).rolling(14).mean()
    rs = up / down.replace(0, np.nan)
    out["rsi14"] = (100 - (100/(1+rs))) / 100.0
    ema12, ema26 = out["Close"].ewm(span=12).mean(), out["Close"].ewm(span=26).mean()
    out["macd"] = (ema12 - ema26) / out["Close"]
    out["vol20"] = out["ret1"].rolling(20).std() * np.sqrt(252)
    out["fwd_ret"] = out["Close"].shift(-HORIZON) / out["Close"] - 1
    return out

TECH_COLS = ["ret1","ret5","rsi14","macd","vol20"]
price_data = {tk: add_technical(fetch_prices(tk)) for tk in TICKERS}
print({tk: len(df) for tk, df in list(price_data.items())[:3]}, "...")

### 2. LLM/SLM sentiment decomposition: polarity x novelty x materiality (real headlines only)

In [ ]:
from transformers import pipeline
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=DEVICE)
POLARITY_SIGN = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}

MATERIALITY_KEYWORDS = ["earnings","profit","loss","guidance","acquisition","merger","stake","ipo",
                         "dividend","buyback","lawsuit","regulatory","rbi","repo","downgrade","upgrade",
                         "rating","fraud","default","ceo","resign","results","revenue","contract","deal"]

def materiality_score(title):
    t = title.lower()
    hits = sum(1 for kw in MATERIALITY_KEYWORDS if kw in t)
    return min(1.0, hits / 3.0)

def add_novelty(df, window=5):
    if len(df) < 2:
        df["novelty"] = 1.0
        return df
    vec = TfidfVectorizer(stop_words="english")
    tfidf = vec.fit_transform(df["title"])
    novelty = []
    for i in range(len(df)):
        lo = max(0, i - window)
        if i == lo:
            novelty.append(1.0); continue
        sims = cosine_similarity(tfidf[i], tfidf[lo:i])
        novelty.append(float(1.0 - sims.max()))
    df["novelty"] = novelty
    return df

def fetch_news_sentiment(ticker):
    try:
        news = yf.Ticker(ticker).news
    except Exception:
        news = []
    rows = []
    for n in news:
        c = n.get("content", {})
        title, pub = c.get("title"), c.get("pubDate")
        if title and pub:
            rows.append({"date": pd.to_datetime(pub).tz_localize(None), "title": title})
    if not rows:
        return pd.DataFrame(columns=["date","title","polarity","novelty","materiality","gate"])
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    sentiments = finbert(df["title"].tolist())
    df["polarity"] = [POLARITY_SIGN[s["label"]] * s["score"] for s in sentiments]
    df["materiality"] = df["title"].apply(materiality_score)
    df = add_novelty(df)
    df["gate"] = df["polarity"] * df["novelty"] * df["materiality"]
    return df

news_frames = {tk: fetch_news_sentiment(tk) for tk in TICKERS}
n_headlines = sum(len(v) for v in news_frames.values())
print(f"real headlines pooled across {len(TICKERS)} tickers: {n_headlines}")

In [ ]:
daily_gate = {}
for tk in TICKERS:
    df = news_frames[tk]
    if df.empty:
        daily_gate[tk] = pd.DataFrame(columns=["polarity","novelty","materiality","gate"])
        continue
    daily_gate[tk] = df.groupby(df["date"].dt.floor("D"))[["polarity","novelty","materiality","gate"]].mean()

rows = []
for tk in TICKERS:
    if daily_gate[tk].empty:
        continue
    merged = price_data[tk][TECH_COLS + ["fwd_ret"]].join(daily_gate[tk], how="inner").dropna()
    merged["ticker"] = tk
    if len(merged):
        rows.append(merged)

panel = pd.concat(rows).sort_index() if rows else pd.DataFrame()
print("sentiment-covered, feature-complete rows:", len(panel), "across", panel['ticker'].nunique() if len(panel) else 0, "tickers")
if len(panel) < 60:
    print("NOTE: this is a small sample -- a known, disclosed constraint of using free news sources over a short "
          "real-time window rather than a paid multi-year archive. Treat results as illustrative of the method, "
          "not as a statistically powered backtest.")
panel.tail(3)

### 3. Train / validation / test split

Because real free news only covers the last few weeks, there usually is not enough *calendar* time for a
meaningful walk-forward split (unlike Notebook 1's multi-year technical backbone). We use a **random
row-level split** here instead and say so explicitly — this is a real methodological difference from the
underlying research's production evaluation (which uses a long, purely time-ordered live ledger), not
something to gloss over in a presentation.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = TECH_COLS + ["gate"]
X = panel[FEATURES].values
y = panel["fwd_ret"].values

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=SEED)
print("train/val/test sizes:", len(X_train), len(X_val), len(X_test))

### 4. Quantile forecaster + a tiny hand-rolled PSO over its hyperparameters

In [ ]:
QUANTILES = [0.05, 0.5, 0.95]

def fit_quantile_models(Xtr, ytr, lr, num_leaves, n_estimators=150):
    models = {}
    for q in QUANTILES:
        m = lgb.LGBMRegressor(objective="quantile", alpha=q, learning_rate=lr,
                               num_leaves=max(3, int(round(num_leaves))), n_estimators=n_estimators,
                               min_child_samples=5, verbose=-1, random_state=SEED)
        m.fit(Xtr, ytr)
        models[q] = m
    return models

def pinball_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return np.mean(np.maximum(q*diff, (q-1)*diff))

def objective(params, Xtr, ytr, Xv, yv):
    lr, num_leaves = params
    lr = float(np.clip(lr, 0.01, 0.3))
    models = fit_quantile_models(Xtr, ytr, lr, num_leaves)
    total = 0.0
    for q, m in models.items():
        total += pinball_loss(yv, m.predict(Xv), q)
    return total / len(models)

class Particle:
    def __init__(self, bounds):
        self.pos = np.array([np.random.uniform(lo, hi) for lo, hi in bounds])
        self.vel = np.zeros(len(bounds))
        self.best_pos, self.best_val = self.pos.copy(), np.inf

def pso(bounds, n_particles=6, n_iters=6, w=0.5, c1=1.5, c2=1.5):
    particles = [Particle(bounds) for _ in range(n_particles)]
    gbest_pos, gbest_val = None, np.inf
    for it in range(n_iters):
        for p in particles:
            val = objective(p.pos, X_train, y_train, X_val, y_val)
            if val < p.best_val:
                p.best_val, p.best_pos = val, p.pos.copy()
            if val < gbest_val:
                gbest_val, gbest_pos = val, p.pos.copy()
        for p in particles:
            r1, r2 = np.random.rand(len(bounds)), np.random.rand(len(bounds))
            p.vel = w*p.vel + c1*r1*(p.best_pos - p.pos) + c2*r2*(gbest_pos - p.pos)
            p.pos = np.clip(p.pos + p.vel, [b[0] for b in bounds], [b[1] for b in bounds])
        print(f"PSO iter {it}: best val pinball loss {gbest_val:.5f}  params(lr, num_leaves)={gbest_pos}")
    return gbest_pos, gbest_val

BOUNDS = [(0.01, 0.3), (7, 63)]
best_params, best_val = pso(BOUNDS)
best_lr, best_leaves = best_params
print(f"\nchosen hyperparameters: learning_rate={best_lr:.4f}, num_leaves={int(round(best_leaves))}")

In [ ]:
final_models = fit_quantile_models(X_train, y_train, best_lr, best_leaves, n_estimators=300)
q_preds_test = {q: final_models[q].predict(X_test) for q in QUANTILES}
q_preds_val  = {q: final_models[q].predict(X_val)  for q in QUANTILES}

lo_test, med_test, hi_test = q_preds_test[0.05], q_preds_test[0.5], q_preds_test[0.95]
lo_val, med_val, hi_val = q_preds_val[0.05], q_preds_val[0.5], q_preds_val[0.95]
raw_coverage = np.mean((y_test >= lo_test) & (y_test <= hi_test))
print(f"raw nominal-90% interval, empirical test coverage: {raw_coverage:.1%}  (target: 90%)")

### 5. Two-stage recalibration — bias fix, then interval-coverage fix

This project's production pipeline found that raw model output needed **two separate repairs**: a systematic
bias in the point forecast (a "stacking domain mismatch"), fixed first, and *then* an interval-coverage
problem, fixed second. We reproduce that same two-stage structure here at demo scale:

- **Stage A — isotonic bias correction**: fit a monotonic map from the raw median forecast to the actual
  outcome, using the **validation** split only, then apply it to the test median (and carry the same shift
  into the low/high quantiles so the interval moves with it).
- **Stage B — conformalized quantile calibration** (Romano et al. 2019): fix the *width* of the
  bias-corrected interval so its empirical coverage matches the nominal 90% on genuinely unseen test data.

In [ ]:
from sklearn.isotonic import IsotonicRegression

bias_corrector = IsotonicRegression(out_of_bounds="clip").fit(med_val, y_val)

def apply_bias_correction(lo, med, hi):
    med_c = bias_corrector.predict(med)
    delta = med_c - med
    return lo + delta, med_c, hi + delta

lo_val_bc, med_val_bc, hi_val_bc = apply_bias_correction(lo_val, med_val, hi_val)
lo_test_bc, med_test_bc, hi_test_bc = apply_bias_correction(lo_test, med_test, hi_test)

raw_bias_mae = np.mean(np.abs(med_val - y_val))
corrected_bias_mae = np.mean(np.abs(med_val_bc - y_val))
print(f"stage A (isotonic bias correction), validation median-forecast MAE: "
      f"{raw_bias_mae:.5f} -> {corrected_bias_mae:.5f}")

In [ ]:
conformity = np.maximum(lo_val_bc - y_val, y_val - hi_val_bc)
n_val = len(conformity)
alpha = 0.10
q_level = min(1.0, np.ceil((n_val+1)*(1-alpha))/n_val)
correction = np.quantile(conformity, q_level)
print(f"stage B conformal correction term: {correction:+.5f}")

lo_cal, hi_cal = lo_test_bc - max(correction, 0), hi_test_bc + max(correction, 0)
cal_coverage = np.mean((y_test >= lo_cal) & (y_test <= hi_cal))
print(f"raw interval, empirical test coverage:              {raw_coverage:.1%}  (target: 90%)")
print(f"bias-corrected + conformal interval, test coverage:  {cal_coverage:.1%}  (target: 90%)")
print(f"raw interval mean width: {np.mean(hi_test-lo_test):.5f}  |  final calibrated mean width: {np.mean(hi_cal-lo_cal):.5f}")

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(["raw", "bias-corrected + conformal"], [raw_coverage, cal_coverage], color=["#c0392b","#2471a3"])
ax.axhline(0.90, color="black", ls="--", lw=1, label="nominal 90% target")
ax.set_ylim(0,1); ax.set_ylabel("empirical coverage"); ax.legend(); ax.set_title("Interval calibration: before vs after (2-stage)")
plt.tight_layout(); plt.show()

### 6. Conviction / abstention gate — trade coverage for accuracy, honestly measured

This mirrors the production result's structure directly: report the no-gate base hit rate, then the hit rate
at a high-conviction operating point **with a Wilson confidence interval** (a real, standard binomial CI —
more honest than a bare point estimate on a small test set), across the full coverage curve.

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    phat = k / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2*n)) / denom
    margin = (z / denom) * np.sqrt(phat*(1-phat)/n + z**2/(4*n**2))
    return max(0.0, center - margin), min(1.0, center + margin)

interval_width = hi_cal - lo_cal
dir_pred = np.sign(med_test_bc)
dir_actual = np.sign(y_test)
base_hit_rate = (dir_pred == dir_actual).mean()
base_lo, base_hi = wilson_ci((dir_pred == dir_actual).sum(), len(y_test))

thresholds = np.linspace(0, 95, 20)
coverages, hit_rates, ns = [], [], []
for pct in thresholds:
    cutoff = np.percentile(interval_width, 100 - pct)   # abstain on the widest `pct`% of intervals
    keep = interval_width <= cutoff
    if keep.sum() < 5:
        continue
    coverages.append(keep.mean())
    hit_rates.append((dir_pred[keep] == dir_actual[keep]).mean())
    ns.append(int(keep.sum()))

print(f"base hit rate, no abstention (n={len(y_test)}): {base_hit_rate:.1%}  (95% CI {base_lo:.1%}-{base_hi:.1%})")

# headline high-conviction point: closest available coverage to the top ~20% most-confident predictions
# (not this project's production ~6%/94%-abstention regime -- our test set is far too small for that to be
# anything but noise; we pick the most extreme threshold this sample size can still support meaningfully)
if coverages:
    target_coverage = min(coverages, key=lambda c: abs(c - 0.20))
    i = coverages.index(target_coverage)
    k = int(round(hit_rates[i] * ns[i]))
    hi_lo, hi_hi = wilson_ci(k, ns[i])
    print(f"high-conviction point: hit rate {hit_rates[i]:.1%} at {coverages[i]:.1%} coverage "
          f"(n={ns[i]}, 95% CI {hi_lo:.1%}-{hi_hi:.1%})")
    print("For scale: this project's production live ledger reports 60.6% hit rate vs a 58.0% base rate at "
          "~94% abstention (95% CI 58.8-62.4) on 619-621 resolved forecasts. A demo-scale pull of a few hundred "
          "rows cannot reach that abstention level with a non-trivial n -- compare the *shape* of the curve "
          "(does the hit rate rise as you abstain more?), not the exact percentages.")

plt.figure(figsize=(7,4))
plt.plot(coverages, hit_rates, marker="o")
plt.axhline(base_hit_rate, color="gray", ls="--", label=f"no-gate baseline ({base_hit_rate:.1%})")
plt.xlabel("fraction of predictions kept (coverage)"); plt.ylabel("directional hit rate")
plt.title("Conviction gate: hit rate vs. how much you're willing to abstain")
plt.legend(); plt.tight_layout(); plt.show()

## Summary for a presentation slide

- **Data**: real NSE prices (yfinance) + real, current news headlines (Yahoo Finance) scored by a real
  financial SLM (FinBERT) — no simulated headlines or invented sentiment scores.
- **Method**: polarity x novelty x materiality multiplicative gate -> LightGBM quantile forecaster (21-day
  swing horizon) -> **two-stage recalibration** (isotonic bias fix, then conformalized interval calibration)
  -> conviction/abstention gate with a Wilson-CI headline number. This is a faithful, compressed version of
  the underlying research's mechanism (same horizon, same two-repair calibration structure, same
  base-rate-vs-gated-rate-with-CI reporting), minus its production-scale live ledger.
- **Architecture-optimization nod**: a 6-particle/6-iteration PSO tunes the forecaster's learning rate and
  tree complexity by minimizing validation pinball loss (the paper-8 idea, at demo scale).
- **Headline numbers to quote**: the before/after coverage bars (Section 5) and the hit-rate-vs-coverage curve
  with its high-conviction Wilson CI (Section 6), taken directly from this run's printed output.
- **Disclose, don't hide**: sample size is modest (free real-time news only) and the split is random rather
  than time-ordered — say this out loud in a presentation rather than presenting the numbers as a large-sample,
  walk-forward result. Compare shape (does the curve rise?), not exact percentages, against the production
  ledger's own 58.0%/60.6%/94%-abstention result.